# Predict Customer Churn - Kaggle Inference & Submission

このノートブックは Kaggle Notebook 環境で実行されることを想定しています。

以下の処理を行います：
1. Kaggle Datasets から訓練済みモデルを読込
2. テストデータに対して推論を実行
3. `submission.csv` を生成して提出

---

In [ ]:
# ============================================================
# 🎯 INFERENCE CONFIGURATION（推論設定）
# ============================================================

# 訓練済みモデルが保存された Kaggle Dataset 名
MODEL_DATASET_NAME = "kaggle-customer-churn-exp001-child-exp000"

# 出力ファイル
OUTPUT_CSV = "submission.csv"

print(f"Model Dataset: {MODEL_DATASET_NAME}")
print(f"Output CSV: {OUTPUT_CSV}")

In [ ]:
# 必要なライブラリをインストール（Kaggle環境での処理）
import subprocess
import sys

packages = ['lightgbm', 'xgboost', 'scikit-learn', 'pandas', 'numpy', 'pyyaml']

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg} already installed")
    except:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [ ]:
# データの確認
import os
import pandas as pd
from pathlib import Path

# Kaggleコンペティションのデータにアクセス
train_df = pd.read_csv('/kaggle/input/predict-customer-churn/train.csv')
test_df = pd.read_csv('/kaggle/input/predict-customer-churn/test.csv')
sample_submission = pd.read_csv('/kaggle/input/predict-customer-churn/sample_submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sample submission shape: {sample_submission.shape}")
print(f"\nSample submission format:")
print(sample_submission.head())

In [ ]:
# Kaggle Datasets から訓練済みモデルをダウンロード
import subprocess
import os
from pathlib import Path

# モデルディレクトリ
model_dir = Path("/kaggle/input") / MODEL_DATASET_NAME

if model_dir.exists():
    print(f"✓ Model directory found: {model_dir}")
    # ファイルを確認
    for f in model_dir.glob("*"):
        print(f"  - {f.name}")
else:
    print(f"⚠ Model directory not found: {model_dir}")
    print(f"\nNote: In Kaggle Notebook, ensure the dataset is added via 'Add input data'")

In [ ]:
# OOF予測を読込（参考：訓練時の精度確認用）
import json

oof_pred_path = model_dir / "oof_predictions.csv"
results_path = model_dir / "results.json"

if oof_pred_path.exists():
    oof_df = pd.read_csv(oof_pred_path)
    print(f"✓ OOF predictions loaded: {oof_df.shape}")
    print(oof_df.head())

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"\n✓ Training results:")
    for key, value in results.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

In [ ]:
# 訓練時と同じ前処理をテストデータに適用
from sklearn.preprocessing import LabelEncoder
import numpy as np
import yaml

# 設定を読込
config_path = model_dir / "config.yaml"
if config_path.exists():
    with open(config_path) as f:
        config = yaml.safe_load(f)
    print(f"✓ Config loaded from {config_path}")
else:
    print(f"⚠ Config not found, using defaults")
    config = {'target': {'name': 'Churn'}, 'training': {'eval_metric': 'auc'}}

# ターゲット列名
target_name = config['target']['name']

# 訓練データから特徴を抽出
y_train = train_df[target_name].values
X_train = train_df.drop(columns=[target_name])
X_test = test_df.copy()

# カテゴリカル特徴のラベルエンコーディング
categorical_cols = X_train.select_dtypes(include='object').columns
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    le_dict[col] = le
    
    # Test は Train の encoder を使う
    test_values = X_test[col].astype(str)
    test_values = np.where(
        test_values.isin(le.classes_),
        le.transform(test_values),
        -1  # unknown カテゴリ
    )
    X_test[col] = test_values

# 欠損値を訓練データの平均値で埋める
train_mean = X_train.mean(numeric_only=True)
X_train = X_train.fillna(train_mean)
X_test = X_test.fillna(train_mean)

print(f"✓ Data preprocessing completed")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape: {X_test.shape}")

In [ ]:
# テスト予測を読込（train.py で事前に生成されている場合）
test_pred_path = model_dir / "test_predictions.csv"

if test_pred_path.exists():
    print(f"✓ Using pre-computed test predictions")
    test_pred_df = pd.read_csv(test_pred_path)
    test_predictions = test_pred_df['prediction'].values
else:
    print(f"⚠ test_predictions.csv not found")
    print(f"   Generating predictions from scratch...")
    
    # モデルファイルを探す
    model_files = list(model_dir.glob("*.pkl")) + list(model_dir.glob("*.joblib"))
    if model_files:
        print(f"   Found model files: {[f.name for f in model_files]}")
        # ここで推論を実行（実装は省略）
    else:
        print(f"   ERROR: No model files found")
        test_predictions = None

if test_predictions is not None:
    print(f"\nTest predictions:")
    print(f"  Min: {test_predictions.min():.4f}")
    print(f"  Max: {test_predictions.max():.4f}")
    print(f"  Mean: {test_predictions.mean():.4f}")

In [ ]:
# Kaggle提出形式で結果を保存
import pandas as pd

if test_predictions is not None:
    # sample_submission を参考にして作成
    submission_df = pd.DataFrame({
        'Id': range(len(test_predictions)),  # ID列の名前はサンプルに合わせる
        'Churn': test_predictions  # ター出列の名前もサンプルに合わせる
    })
    
    # 実際のサンプルフォーマットに合わせる
    if 'sample_id' in sample_submission.columns:
        submission_df.columns = ['sample_id', 'target']
        if len(test_df) > 0 and 'Id' in test_df.columns:
            submission_df['sample_id'] = test_df['Id'].values
    
    # 確率が [0, 1] 範囲内か確認
    assert submission_df[submission_df.columns[-1]].min() >= 0, "Predictions have values < 0"
    assert submission_df[submission_df.columns[-1]].max() <= 1, "Predictions have values > 1"
    
    # 保存
    submission_df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"✓ Submission saved to {OUTPUT_CSV}")
    print(f"  Shape: {submission_df.shape}")
    print(f"\nFirst 10 rows:")
    print(submission_df.head(10))
else:
    print("Cannot create submission: test_predictions not available")

In [ ]:
# 📊 提出前の確認
print(f"""
======================================
提出前チェックリスト
======================================

✓ submission.csv が {OUTPUT_CSV} に保存されました
✓ ファイルサイズを確認してください
✓ 予測値が [0, 1] 範囲内か確認してください
✓ サンプル行数とテストデータが一致するか確認

次のステップ：
1. "Output" タブから {OUTPUT_CSV} をダウンロード
2. または、Kaggleノートブック画面から直接提出

======================================
""")